#  lgb model of AM-II by transfer learning of from AM-I 

In [6]:
import os
import json
import re
import joblib
import logging
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from lightgbm import early_stopping

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error


# ============================================================
# 1) Deterministic / Reproducible Setup
#    - Fix ALL RNG seeds (python / numpy / optuna / lightgbm)
#    - Force LightGBM deterministic behavior
#    - Use stable CV splits (already fixed by KFold random_state)
# ============================================================
SEED = 42

def set_global_determinism(seed: int = 42, num_threads: int = 20):
    # python
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)

    # numpy
    np.random.seed(seed)

    # lightgbm threads
    os.environ["LIGHTGBM_NUM_THREADS"] = str(num_threads)
    # optional: limit other BLAS threads if you use numpy/scipy heavy ops elsewhere
    os.environ.setdefault("OMP_NUM_THREADS", str(num_threads))
    os.environ.setdefault("MKL_NUM_THREADS", str(num_threads))


set_global_determinism(SEED, num_threads=20)

# -------------------- Global Settings --------------------
DATA_DIR = "./1-train_test_split"
PRETRAIN_MODEL_DIR = "./2-lgb-models"

OUTPUT_FOLDER = "./"
MODEL_DIR = os.path.join(OUTPUT_FOLDER, "2-lgb-models-TL")
os.makedirs(MODEL_DIR, exist_ok=True)

# -------------------- Logging Configuration --------------------
log_file_path = os.path.join(MODEL_DIR, "AMII-lgb_transfer_from_AM-I.log")
logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="a"
)

TARGET_DATASETS = [
    "AM-II-filtered_with_labels_k4",
]

PRETRAIN_TAG = "AM-I-filtered_with_labels_k4"
PRETRAIN_SAFE_TAG = PRETRAIN_TAG  # keep original

PRETRAIN_MODEL_PATH = os.path.join(PRETRAIN_MODEL_DIR, f"{PRETRAIN_SAFE_TAG}_model.joblib")
PRETRAIN_FEATURE_PATH = os.path.join(PRETRAIN_MODEL_DIR, f"{PRETRAIN_SAFE_TAG}_feature_list.pkl")
PRETRAIN_IMPUTER_PATH = os.path.join(PRETRAIN_MODEL_DIR, f"{PRETRAIN_SAFE_TAG}_imputer.pkl")
PRETRAIN_METRICS_PATH = os.path.join(PRETRAIN_MODEL_DIR, f"{PRETRAIN_SAFE_TAG}_metrics.json")

IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#AEAEB2",
    "residual": "#34C759",
    "text": "#000000"
}

def iphone_style_ax(ax):
    ax.tick_params(axis="both", direction="out", length=6, width=2, labelsize=16)
    for spine in ["top", "right", "bottom", "left"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)

def safe_tag(tag: str) -> str:
    return tag


# ============================================================
# 2) Plotting Functions (unchanged behavior)
# ============================================================
def plot_scatter_and_residuals(y_true, y_pred, save_folder, base_name):
    try:
        os.makedirs(save_folder, exist_ok=True)

        # Scatter
        plt.figure(figsize=(6, 6))
        ax = plt.gca()
        iphone_style_ax(ax)
        ax.set_aspect("equal", adjustable="box")

        plt.scatter(
            y_true, y_pred,
            alpha=0.8,
            s=70,
            color=IPHONE_COLORS["scatter"],
            edgecolors="none"
        )

        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        plt.plot(
            lims, lims,
            linestyle="--",
            color=IPHONE_COLORS["line"],
            linewidth=3
        )

        r2 = r2_score(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)

        plt.xlabel("True RT (s)", fontsize=18, fontweight="bold")
        plt.ylabel("Predicted RT (s)", fontsize=18, fontweight="bold")

        plt.text(
            0.05, 0.95,
            f"R² = {r2:.3g}\nMAE = {mae:.3g}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=16,
            color=IPHONE_COLORS["text"]
        )

        plt.tight_layout()
        plt.savefig(os.path.join(save_folder, f"{base_name}_scatter.png"), dpi=600)
        plt.close()

        # Residual
        residuals = y_pred - y_true
        plt.figure(figsize=(6, 6))
        ax = plt.gca()
        iphone_style_ax(ax)

        plt.scatter(y_pred, residuals, alpha=0.6, color=IPHONE_COLORS["scatter"], edgecolor="k", s=50)
        plt.axhline(y=0, linestyle="--", color=IPHONE_COLORS["line"], linewidth=2)

        plt.xlabel("Predicted Retention Time (s)", fontsize=18, fontweight="bold")
        plt.ylabel("Residuals (Predicted - True)", fontsize=18, fontweight="bold")

        rmin, rmax = float(np.min(residuals)), float(np.max(residuals))
        pad = 0.1 * (rmax - rmin + 1e-9)
        ax.set_ylim([rmin - pad, rmax + pad])

        plt.tight_layout()
        plt.savefig(os.path.join(save_folder, f"{base_name}_residuals.png"), dpi=600)
        plt.close()

    except Exception as e:
        logging.error(f"[{base_name}] Plotting failed: {str(e)}", exc_info=True)


def plot_optuna_rmse(study, save_path):
    try:
        best_rmses = [t.value for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if not best_rmses:
            return
        cumulative_best = np.minimum.accumulate(best_rmses)
        plt.figure(figsize=(6, 4))
        plt.plot(cumulative_best, marker="o", linestyle="-", color="blue")
        plt.xlabel("Trial")
        plt.ylabel("Best RMSE so far")
        plt.title("Optuna RMSE Convergence")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(save_path, dpi=600)
        plt.close()
    except Exception as e:
        logging.error(f"Failed to plot Optuna RMSE curve: {str(e)}", exc_info=True)


# ============================================================
# 3) Pretrained Asset Loading
# ============================================================
def load_pretrained_assets():
    if not (os.path.isfile(PRETRAIN_MODEL_PATH) and os.path.isfile(PRETRAIN_FEATURE_PATH) and os.path.isfile(PRETRAIN_IMPUTER_PATH)):
        raise FileNotFoundError("Pretrained assets not found")

    model = joblib.load(PRETRAIN_MODEL_PATH)
    feature_cols = joblib.load(PRETRAIN_FEATURE_PATH)
    imputer = joblib.load(PRETRAIN_IMPUTER_PATH)

    pretrain_params = {}
    if os.path.isfile(PRETRAIN_METRICS_PATH):
        try:
            with open(PRETRAIN_METRICS_PATH, "r") as f:
                pretrain_params = json.load(f).get("best_params", {})
        except Exception:
            pass

    return model, feature_cols, imputer, pretrain_params


# ============================================================
# 4) Data Checking (unchanged)
# ============================================================
def check_and_extract(df: pd.DataFrame, feature_cols: list, target_col: str):
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Data missing the following feature columns: {missing}")
    if target_col not in df.columns:
        raise KeyError(f"Missing target column '{target_col}'")
    X = df[feature_cols].values
    y = df[target_col].values
    return X, y


# ============================================================
# 5) Fine-tuning with Optuna (deterministic + no leakage)
#    Key changes for reproducibility:
#      - Optuna sampler seeded
#      - LightGBM deterministic params + fixed seeds
#      - Force row-wise to reduce non-determinism
#      - Fix n_jobs/num_threads
#      - Use booster from pretrained model as init_model (no mutation risk)
#    Avoid data leakage:
#      - NO use of X_test/y_test in Optuna/CV
#      - Imputer is applied as-is (pretrained). If你怀疑它在预训练时用了测试集，
#        应在预训练流程里修正；这里不会引入目标数据集的 test 泄露。
# ============================================================
def finetune_with_optuna(
    tag_raw,
    pretrain_model,
    feature_cols,
    imputer,
    pretrain_params,
    target_col="UV_RT-s",
    n_trials=40,
    n_splits=5,
    new_trees=50
):
    tag = safe_tag(tag_raw)
    train_path = os.path.join(DATA_DIR, f"{tag_raw}_train.csv")
    test_path = os.path.join(DATA_DIR, f"{tag_raw}_test.csv")

    dataset_dir = os.path.join(MODEL_DIR, f"{tag}")
    os.makedirs(dataset_dir, exist_ok=True)

    if not os.path.isfile(train_path) or not os.path.isfile(test_path):
        msg = f"Cannot find train/test CSV for {tag_raw}"
        logging.warning(msg)
        return {"tag": tag_raw, "status": "warn", "message": msg}

    # Read
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Extract
    X_all_raw, y_all = check_and_extract(train_df, feature_cols, target_col)
    X_test_raw, y_test = check_and_extract(test_df, feature_cols, target_col)

    # Transform using pretrained imputer (no test leakage in THIS script)
    X_all = imputer.transform(X_all_raw)
    X_test = imputer.transform(X_test_raw)

    # -------------------- Base Params --------------------
    base_params = pretrain_model.get_params()

    # remove unstable keys and re-add in a controlled way
    base_params.pop("n_estimators", None)
    base_params.pop("random_state", None)
    base_params.pop("n_jobs", None)

    if pretrain_params:
        pretrain_params.pop("n_estimators", None)
        pretrain_params.pop("random_state", None)
        pretrain_params.pop("n_jobs", None)
        base_params.update(pretrain_params)

    # -------------------- Deterministic LightGBM Defaults --------------------
    # These are critical to ensure repeated runs are identical.
    deterministic_defaults = {
        # Determinism & seeds
        "deterministic": True,
        "random_state": SEED,
        "data_random_seed": SEED,
        "feature_fraction_seed": SEED,
        "bagging_seed": SEED,
        "drop_seed": SEED,

        # Reduce parallel non-determinism
        "force_row_wise": True,

        # Threads
        "n_jobs": int(os.environ.get("LIGHTGBM_NUM_THREADS", "20")),
    }

    # Note: keep user/pretrain settings, but enforce determinism keys
    base_params.update(deterministic_defaults)

    # init_model: prefer Booster to avoid accidental wrapper state issues
    init_booster = getattr(pretrain_model, "booster_", None)
    init_model_for_fit = init_booster if init_booster is not None else pretrain_model

    # -------------------- Multi-stage Parameter Tuning --------------------
    def objective(trial, stage="coarse"):
        params = base_params.copy()

        if stage == "coarse":
            params.update({
                "learning_rate": trial.suggest_float("learning_rate", 0.002, 0.1, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 16, 128),
                "max_depth": trial.suggest_int("max_depth", 3, 12),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            })
        else:
            params.update({
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
                "num_leaves": trial.suggest_int(
                    "num_leaves",
                    max(16, params.get("num_leaves", 31) - 16),
                    min(128, params.get("num_leaves", 31) + 16)
                ),
                "max_depth": trial.suggest_int(
                    "max_depth",
                    max(3, params.get("max_depth", 7) - 3),
                    min(12, params.get("max_depth", 7) + 3)
                ),
            })

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        rmses = []

        for train_idx, valid_idx in kf.split(X_all):
            X_tr, X_val = X_all[train_idx], X_all[valid_idx]
            y_tr, y_val = y_all[train_idx], y_all[valid_idx]

            model = lgb.LGBMRegressor(
                **params,
                n_estimators=int(getattr(pretrain_model, "n_estimators", 1000)) + int(new_trees),
            )

            # Important: only eval on CV-valid split, never touch X_test here (avoid leakage)
            model.fit(
                X_tr, y_tr,
                init_model=init_model_for_fit,
                eval_set=[(X_val, y_val)],
                eval_metric="rmse",
                callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
            )

            y_pred = model.predict(X_val)
            rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

        return float(np.mean(rmses))

    # -------------------- Optuna (seeded) --------------------
    sampler = TPESampler(seed=SEED)
    pruner = MedianPruner(n_startup_trials=5)

    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    study.optimize(lambda t: objective(t, stage="coarse"), n_trials=n_trials, show_progress_bar=True)
    plot_optuna_rmse(study, os.path.join(dataset_dir, f"{tag}_coarse_optuna.png"))

    # -------------------- Train Final Models --------------------
    best_params = base_params.copy()
    best_params.update(study.best_params)

    n_estimators_final = int(getattr(pretrain_model, "n_estimators", 1000)) + int(new_trees)

    # Fine-tuned (transfer) model
    fine_model = lgb.LGBMRegressor(
        **best_params,
        n_estimators=n_estimators_final,
    )
    fine_model.fit(
        X_all, y_all,
        init_model=init_model_for_fit
    )

    # Scratch model (same hyperparams, no init_model)
    scratch_model = lgb.LGBMRegressor(
        **best_params,
        n_estimators=n_estimators_final,
    )
    scratch_model.fit(X_all, y_all)

    # -------------------- Test Evaluation (only here) --------------------
    y_pred_test_fine = fine_model.predict(X_test)
    y_pred_test_scratch = scratch_model.predict(X_test)

    rmse_fine = np.sqrt(mean_squared_error(y_test, y_pred_test_fine))
    r2_fine = r2_score(y_test, y_pred_test_fine)
    mae_fine = mean_absolute_error(y_test, y_pred_test_fine)

    rmse_scratch = np.sqrt(mean_squared_error(y_test, y_pred_test_scratch))
    r2_scratch = r2_score(y_test, y_pred_test_scratch)
    mae_scratch = mean_absolute_error(y_test, y_pred_test_scratch)

    plot_scatter_and_residuals(y_test, y_pred_test_fine, dataset_dir, f"{tag}_finetuned")
    plot_scatter_and_residuals(y_test, y_pred_test_scratch, dataset_dir, f"{tag}_scratch")

    # -------------------- Save Models and Parameters --------------------
    fine_model_path = os.path.join(dataset_dir, f"{tag}_finetuned_model.joblib")
    scratch_model_path = os.path.join(dataset_dir, f"{tag}_scratch_model.joblib")
    feature_path = os.path.join(dataset_dir, f"{tag}_feature_list.pkl")
    imputer_path = os.path.join(dataset_dir, f"{tag}_imputer.pkl")
    metrics_path = os.path.join(dataset_dir, f"{tag}_metrics.json")

    joblib.dump(fine_model, fine_model_path)
    joblib.dump(scratch_model, scratch_model_path)
    joblib.dump(feature_cols, feature_path)
    joblib.dump(imputer, imputer_path)

    metrics = {
        "finetuned": {"rmse": float(rmse_fine), "r2": float(r2_fine), "mae": float(mae_fine)},
        "scratch": {"rmse": float(rmse_scratch), "r2": float(r2_scratch), "mae": float(mae_scratch)},
        "best_params": best_params
    }

    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=4)

    logging.info(
        f"[{tag_raw}] Fine-tuning completed. "
        f"Finetuned RMSE={rmse_fine:.6f}, Scratch RMSE={rmse_scratch:.6f}"
    )

    return {
        "tag": tag_raw,
        "status": "ok",
        "metrics": metrics,
        "fine_model_path": fine_model_path,
        "scratch_model_path": scratch_model_path
    }


# ============================================================
# 6) Main
# ============================================================
if __name__ == "__main__":
    try:
        pretrain_model, feature_cols, imputer, pretrain_params = load_pretrained_assets()
    except FileNotFoundError as e:
        logging.error(str(e))
        raise

    results = []
    for target_dataset in TARGET_DATASETS:
        res = finetune_with_optuna(
            target_dataset,
            pretrain_model,
            feature_cols,
            imputer,
            pretrain_params,
            n_trials=40,
            n_splits=5,
            new_trees=50
        )
        results.append(res)

    results_path = os.path.join(MODEL_DIR, "finetune_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=4)

    logging.info("All dataset fine-tuning completed.")


[I 2026-02-12 18:01:26,393] A new study created in memory with name: no-name-20b8afb3-2fd9-4c90-8a2f-8183d7655682
  0%|          | 0/40 [00:00<?, ?it/s]/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-p

[I 2026-02-12 18:01:31,598] Trial 0 finished with value: 4.268917251842011 and parameters: {'learning_rate': 0.008656900442587762, 'num_leaves': 123, 'max_depth': 10, 'min_child_samples': 32, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481}. Best is trial 0 with value: 4.268917251842011.


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.w

[I 2026-02-12 18:01:35,586] Trial 1 finished with value: 5.201505027214045 and parameters: {'learning_rate': 0.002510223034594768, 'num_leaves': 113, 'max_depth': 9, 'min_child_samples': 37, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978}. Best is trial 0 with value: 4.268917251842011.


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.w

...

/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.w

[I 2026-02-12 18:03:11,316] Trial 38 finished with value: 4.082688588325074 and parameters: {'learning_rate': 0.09717372417067749, 'num_leaves': 54, 'max_depth': 6, 'min_child_samples': 30, 'subsample': 0.7434895027147692, 'colsample_bytree': 0.6211646750411223}. Best is trial 11 with value: 3.9342223023518033.


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.w

[I 2026-02-12 18:03:15,233] Trial 39 finished with value: 4.049498987941192 and parameters: {'learning_rate': 0.013819985926258138, 'num_leaves': 69, 'max_depth': 8, 'min_child_samples': 8, 'subsample': 0.691777117202827, 'colsample_bytree': 0.9207849531902829}. Best is trial 11 with value: 3.9342223023518033.


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# lgb model of AM-III, AM-IV, AM-V, and AM-VI by transfering learning from AM-I LGB MODELS

In [2]:
import os
import json
import joblib
import logging
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import RobustScaler
from scipy.stats import mstats


# -------------------- Global Settings --------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "./processed_results"
PRETRAIN_DIR = "./2-lgb-models"
OUTPUT_DIR = "./2-lgb-TL-models-other4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_DATASETS = [
    "AM-III-filtered.csv",
    "AM-IV-filtered.csv",
    "AM-V-filtered.csv",
    "AM-VI-filtered.csv"
]

PRETRAIN_TAG = "AM-I-filtered_with_labels_k4"
# Keep PRETRAIN_TAG as is without sanitization
PRETRAIN_SAFE_TAG = PRETRAIN_TAG

PRETRAIN_MODEL_PATH = os.path.join(PRETRAIN_DIR, f"{PRETRAIN_SAFE_TAG}_model.joblib")
PRETRAIN_FEATURE_PATH = os.path.join(PRETRAIN_DIR, f"{PRETRAIN_SAFE_TAG}_feature_list.pkl")
PRETRAIN_IMPUTER_PATH = os.path.join(PRETRAIN_DIR, f"{PRETRAIN_SAFE_TAG}_imputer.pkl")
PRETRAIN_METRICS_PATH = os.path.join(PRETRAIN_DIR, f"{PRETRAIN_SAFE_TAG}_metrics.json")


# -------------------- Logging Configuration --------------------
logging.basicConfig(
    filename="./2-lgb-TL-models-other4/lgb-TL_nested_cv-other4.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="a"
)

# iPhone color scheme for consistent plotting
IPHONE_COLORS = {
    "scatter": "#007AFF",  # iPhone blue
    "line": "#AEAEB2",     # iPhone gray
    "text": "#000000"      # Black
}

# -------------------- Enhanced Plotting Functions --------------------
def plot_scatter_and_residuals(y_true, y_pred, save_folder, base_name, dpi=300, save=True, 
                               outer_cv_metrics=None, fold_metrics=None):
    """
    绘制高质量的散点图，严格按照指定格式
    新增参数：
        outer_cv_metrics: Outer CV的指标字典（包含平均值和标准差）
        fold_metrics: 各fold的具体指标列表（用于计算总体指标）
    """
    if not save:
        return None, None
    
    os.makedirs(save_folder, exist_ok=True)
    
    # 创建图形，指定尺寸
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    # 轴样式配置
    ax.tick_params(
        axis='both', 
        direction='out', 
        length=6, 
        width=2, 
        labelsize=16
    )
    
    # 显示所有边框
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)
    
    # 无网格
    plt.grid(False)
    
    # 散点图，严格按照指定规则
    plt.scatter(
        y_true, y_pred,
        alpha=0.8,                 # 80% 不透明度
        s=70,                      # 点大小: 70
        color=IPHONE_COLORS['scatter'],  # iPhone蓝色 (#007AFF)
        edgecolors='none'          # 点无边框
    )
    
    # 理想拟合线（对角线）
    # 确保x和y轴范围一致
    xymin = float(min(y_true.min(), y_pred.min()))
    xymax = float(max(y_true.max(), y_pred.max()))
    
    # 计算合适的padding
    data_range = xymax - xymin
    if data_range == 0:
        pad = xymin * 0.1 if xymin != 0 else 1
    else:
        pad = data_range * 0.05  # 5% padding
    
    lim_min = xymin - pad
    lim_max = xymax + pad
    
    # 绘制对角线
    plt.plot(
        [lim_min, lim_max], [lim_min, lim_max],
        linestyle='--',            # 虚线样式
        color=IPHONE_COLORS['line'],  # iPhone灰色 (#AEAEB2)
        linewidth=3                # 线宽: 3
    )
    
    # 计算直接来自数据的指标（用于内部使用）
    r2_direct = r2_score(y_true, y_pred)
    mae_direct = mean_absolute_error(y_true, y_pred)
    
    # 轴标签，加粗字体
    plt.xlabel(
        "True RT (s)", 
        fontsize=18, 
        fontweight='bold'  # 加粗x轴标签
    )
    plt.ylabel(
        "Predicted RT (s)", 
        fontsize=18, 
        fontweight='bold'  # 加粗y轴标签
    )
    
    # 决定显示哪些指标
    if outer_cv_metrics is not None:
        # 使用Outer CV的平均值±标准差格式
        r2_mean, r2_std = outer_cv_metrics.get("R2", (r2_direct, 0))
        mae_mean, mae_std = outer_cv_metrics.get("MAE", (mae_direct, 0))
        
        # 格式化显示：平均值±标准差，保留合适的小数位数
        # 根据数值大小决定显示的小数位数
        r2_std_formatted = f"{r2_std:.2f}" if r2_std < 0.01 else f"{r2_std:.3f}"
        mae_std_formatted = f"{mae_std:.1f}" if mae_std < 0.01 else f"{mae_std:.2f}"
        
        metrics_text = f"R² = {r2_mean:.3f} ± {r2_std_formatted}\nMAE = {mae_mean:.2f} ± {mae_std_formatted}"
        
    elif fold_metrics is not None:
        # 从各fold指标计算平均值和标准差
        r2_values = fold_metrics.get("R2", [])
        mae_values = fold_metrics.get("MAE", [])
        
        if r2_values and mae_values:
            r2_mean, r2_std = np.mean(r2_values), np.std(r2_values)
            mae_mean, mae_std = np.mean(mae_values), np.std(mae_values)
            
            r2_std_formatted = f"{r2_std:.2f}" if r2_std < 0.01 else f"{r2_std:.3f}"
            mae_std_formatted = f"{mae_std:.2f}" if mae_std < 0.01 else f"{mae_std:.2f}"
            
            metrics_text = f"R² = {r2_mean:.3f} ± {r2_std_formatted}\nMAE = {mae_mean:.2f} ± {mae_std_formatted}"
        else:
            metrics_text = f"R² = {r2_direct:.3f}\nMAE = {mae_direct:.2f}"
    else:
        # 默认显示直接计算的指标
        metrics_text = f"R² = {r2_direct:.3f}\nMAE = {mae_direct:.3f}"
    
    # 添加R²和MAE文本到左上角
    plt.text(
        0.05, 0.95,                # 位置: 左上角 (5%, 95%)
        metrics_text,              # 格式化的指标文本
        transform=ax.transAxes, 
        verticalalignment='top',
        fontsize=16, 
        color=IPHONE_COLORS['text'],  # 黑色 (#000000)
        bbox=dict(
            facecolor='white', 
            alpha=0.7, 
            edgecolor='none',
            pad=5
        )
    )
    
    # 设置轴范围一致
    plt.xlim([lim_min, lim_max])
    plt.ylim([lim_min, lim_max])
    
    # 确保紧凑布局并以高分辨率保存
    plt.tight_layout()
    
    # 保存图片
    scatter_path = os.path.join(save_folder, f"{base_name}_outer_cv_scatter.png")
    plt.savefig(scatter_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()
    
    # 记录日志
    if outer_cv_metrics is not None:
        r2_mean, r2_std = outer_cv_metrics.get("R2", (r2_direct, 0))
        mae_mean, mae_std = outer_cv_metrics.get("MAE", (mae_direct, 0))
        logging.info(f"[{base_name}] Outer CV散点图已保存: {scatter_path}")
        logging.info(f"[{base_name}] Outer CV性能指标 - R²: {r2_mean:.4f}±{r2_std:.4f}, MAE: {mae_mean:.4f}±{mae_std:.4f}")
        logging.info(f"[{base_name}] 直接计算的指标 - R²: {r2_direct:.4f}, MAE: {mae_direct:.4f}")
    else:
        logging.info(f"[{base_name}] Outer CV散点图已保存: {scatter_path}")
        logging.info(f"[{base_name}] 性能指标 - R²: {r2_direct:.4f}, MAE: {mae_direct:.4f}")
    
    return r2_direct, mae_direct


# -------------------- Pretrained Assets Loading --------------------
def load_pretrained_assets():
    """加载预训练模型和相关资产"""
    logging.info(f"加载预训练模型: {PRETRAIN_MODEL_PATH}")
    model = joblib.load(PRETRAIN_MODEL_PATH)
    
    logging.info(f"加载特征列: {PRETRAIN_FEATURE_PATH}")
    feature_cols = joblib.load(PRETRAIN_FEATURE_PATH)
    
    logging.info(f"加载Imputer: {PRETRAIN_IMPUTER_PATH}")
    imputer = joblib.load(PRETRAIN_IMPUTER_PATH)
    
    pretrain_params = {}
    if os.path.isfile(PRETRAIN_METRICS_PATH):
        try:
            with open(PRETRAIN_METRICS_PATH, "r") as f:
                pretrain_params = json.load(f).get("best_params", {})
            logging.info("从预训练指标文件中加载参数")
        except Exception as e:
            logging.warning(f"加载预训练参数失败: {e}")
    
    logging.info(f"预训练模型树数量: {model.n_estimators}")
    logging.info(f"特征数量: {len(feature_cols)}")
    
    return model, feature_cols, imputer, pretrain_params


# -------------------- Data Checking --------------------
def check_and_extract(df: pd.DataFrame, feature_cols: list, target_col: str):
    """
    提取并预处理数据，确保一致性
    """
    logging.debug(f"数据提取: 特征数量={len(feature_cols)}, 目标列={target_col}")
    
    # 确保特征列存在
    missing_features = [col for col in feature_cols if col not in df.columns]
    if missing_features:
        logging.warning(f"缺失特征: {missing_features[:5]}")
    
    X = df[feature_cols].values
    y = df[target_col].values
    
    # 确保winsorize处理的一致性
    logging.debug("应用Winsorize处理...")
    for i in range(X.shape[1]):
        X[:, i] = mstats.winsorize(
            X[:, i], 
            limits=[0.01, 0.01],
            nan_policy='omit'
        ).data
    
    y = mstats.winsorize(
        y, 
        limits=[0.01, 0.01],
        nan_policy='omit'
    ).data
    
    # 应用RobustScaler
    logging.debug("应用RobustScaler...")
    scaler = RobustScaler()
    X = scaler.fit_transform(X)
    
    logging.debug(f"处理后数据形状: X={X.shape}, y={y.shape}")
    return X, y


# -------------------- Inner Parameter Tuning --------------------
def optuna_inner_cv(X, y, base_params, pretrain_model, n_trials=20, n_splits=3, new_trees=20):
    """
    使用Optuna进行内层交叉验证参数调优，确保可重复性
    """
    logging.debug(f"开始内层CV调优: n_trials={n_trials}, n_splits={n_splits}, new_trees={new_trees}")
    
    def objective(trial):
        """Optuna目标函数"""
        params = base_params.copy()
        
        # 参数搜索空间
        params.update({
            "learning_rate": trial.suggest_float("learning_rate", 0.002, 0.05, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 64),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 30),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        })
        
        # 使用相同的种子进行KFold分割
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        rmses = []
        
        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X)):
            X_tr, X_val = X[tr_idx], X[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            
            # 创建LightGBM模型，设置所有随机种子
            model = lgb.LGBMRegressor(
                **params,
                n_estimators=pretrain_model.n_estimators + new_trees,
                random_state=SEED,
                force_col_wise=True,
                deterministic=True,  # 确保确定性
                # LightGBM的额外确定性设置
                feature_fraction_seed=SEED,
                bagging_seed=SEED,
                drop_seed=SEED,
                data_random_seed=SEED
            )
            
            # 微调训练，使用预训练模型初始化
            try:
                # 修正：使用callbacks来控制verbose
                model.fit(
                    X_tr, y_tr,
                    init_model=pretrain_model,
                    eval_set=[(X_val, y_val)],
                    eval_metric="rmse",
                    callbacks=[
                        early_stopping(25, verbose=False),
                        log_evaluation(period=0)  # 不显示训练日志
                    ]
                )
                
                # 预测和评估
                y_pred = model.predict(X_val)
                rmse = np.sqrt(mean_squared_error(y_val, y_pred))
                rmses.append(rmse)
                
                trial.set_user_attr(f"fold_{fold_idx}_rmse", rmse)
            except Exception as e:
                logging.error(f"内层CV fold {fold_idx} 训练失败: {e}")
                # 如果训练失败，返回一个很大的值
                return float('inf')
        
        if len(rmses) == 0:
            return float('inf')
            
        avg_rmse = float(np.mean(rmses))
        trial.set_user_attr("avg_rmse", avg_rmse)
        trial.set_user_attr("std_rmse", float(np.std(rmses)))
        
        return avg_rmse
    
    # 创建Optuna研究，使用确定的采样器
    study = optuna.create_study(
        direction="minimize",
        pruner=MedianPruner(
            n_startup_trials=3,
            n_warmup_steps=5
        ),
        sampler=TPESampler(
            seed=SEED,
            n_startup_trials=10,
            multivariate=True,
            group=True
        )
    )
    
    # 优化过程
    logging.info(f"开始Optuna优化，共{n_trials}次试验...")
    study.optimize(
        objective, 
        n_trials=n_trials,
        show_progress_bar=False
    )
    
    if study.best_value == float('inf'):
        logging.warning("Optuna优化失败，所有试验都返回了inf，使用基础参数")
        return base_params.copy()
    
    logging.info(f"Optuna优化完成，最佳RMSE: {study.best_value:.4f}")
    logging.info(f"最佳参数: {study.best_params}")
    
    return study.best_params


# -------------------- Nested CV Main Function --------------------
def nested_cv_finetune(file_name, pretrain_model, feature_cols, imputer, pretrain_params,
                       target_col="UV_RT-s", outer_splits=5, inner_splits=3, n_trials=20, new_trees=20):
    """
    执行嵌套交叉验证，保存Outer CV预测结果和高质量的散点图
    """
    # 提取基础名称
    base_name = file_name.replace('.csv', '')
    dataset_dir = os.path.join(OUTPUT_DIR, base_name)
    os.makedirs(dataset_dir, exist_ok=True)
    
    # 为Outer CV预测创建专门的目录
    outer_cv_dir = os.path.join(dataset_dir, "outer_cv_predictions")
    os.makedirs(outer_cv_dir, exist_ok=True)
    
    logging.info(f"=== 开始数据集 {base_name} 的嵌套CV ===")
    logging.info(f"输出目录: {dataset_dir}")
    logging.info(f"Outer CV splits: {outer_splits}, Inner CV splits: {inner_splits}")
    logging.info(f"Optuna trials: {n_trials}, New trees: {new_trees}")

    # 加载数据
    train_path = os.path.join(DATA_DIR, file_name)
    df = pd.read_csv(train_path)
    logging.info(f"数据加载: {train_path}, 形状: {df.shape}")
    
    # 检查目标列是否存在
    if target_col not in df.columns:
        raise ValueError(f"目标列 '{target_col}' 不在数据集中")
    
    # 数据预处理
    X_all, y_all = check_and_extract(df, feature_cols, target_col)
    X_all = imputer.transform(X_all)
    logging.info(f"预处理后: X形状={X_all.shape}, y形状={y_all.shape}")
    
    # 准备基础参数
    base_params = pretrain_model.get_params()
    base_params.pop("n_estimators", None)
    base_params.pop("random_state", None)
    
    if pretrain_params:
        pretrain_params.pop("n_estimators", None)
        pretrain_params.pop("random_state", None)
        base_params.update(pretrain_params)
    
    # Outer CV
    kf_outer = KFold(n_splits=outer_splits, shuffle=True, random_state=SEED)
    rmse_list, r2_list, mae_list = [], [], []
    
    # 存储所有Outer CV的预测结果
    all_true = []
    all_pred = []
    all_fold_indices = []
    all_fold_numbers = []
    
    for fold, (train_idx, test_idx) in enumerate(kf_outer.split(X_all)):
        logging.info(f"--- Outer Fold {fold+1}/{outer_splits} ---")
        
        X_tr, X_te = X_all[train_idx], X_all[test_idx]
        y_tr, y_te = y_all[train_idx], y_all[test_idx]
        
        logging.debug(f"训练集: {X_tr.shape}, 测试集: {X_te.shape}")

        # Inner parameter tuning
        logging.info(f"开始内层参数调优...")
        best_inner_params = optuna_inner_cv(
            X_tr, y_tr, base_params, pretrain_model,
            n_trials=n_trials, n_splits=inner_splits, 
            new_trees=new_trees
        )
        
        # 合并参数
        params_final = base_params.copy()
        params_final.update(best_inner_params)
        
        logging.info(f"内层调优完成，使用参数进行训练")

        # 使用微调后的参数训练模型
        model = lgb.LGBMRegressor(
            **params_final,
            n_estimators=pretrain_model.n_estimators + new_trees,
            random_state=SEED,
            force_col_wise=True,
            deterministic=True,
            feature_fraction_seed=SEED,
            bagging_seed=SEED,
            drop_seed=SEED,
            data_random_seed=SEED
        )
        
        # 微调训练 - 修正：移除verbose参数
        model.fit(
            X_tr, y_tr, 
            init_model=pretrain_model,
            callbacks=[log_evaluation(period=0)]  # 不显示日志
        )
        
        # 预测
        y_pred = model.predict(X_te)
        
        # 存储当前fold的预测结果
        all_true.extend(y_te)
        all_pred.extend(y_pred)
        all_fold_indices.extend(test_idx)
        all_fold_numbers.extend([fold] * len(y_te))

        # 计算指标
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        r2 = r2_score(y_te, y_pred)
        mae = mean_absolute_error(y_te, y_pred)
        
        rmse_list.append(rmse)
        r2_list.append(r2)
        mae_list.append(mae)

        logging.info(f"[Outer Fold {fold+1}] RMSE={rmse:.4f}, R2={r2:.4f}, MAE={mae:.4f}")

    # 保存所有Outer CV预测结果到CSV
    logging.info("保存Outer CV预测结果...")
    predictions_df = pd.DataFrame({
        'fold': all_fold_numbers,
        'original_index': all_fold_indices,
        'true_value': all_true,
        'predicted_value': all_pred
    })
    
    # 按原始数据顺序排序
    predictions_df = predictions_df.sort_values('original_index')
    
    # 添加原始数据信息
    original_data = df.copy()
    original_data['outer_cv_prediction'] = np.nan
    original_data.loc[predictions_df['original_index'], 'outer_cv_prediction'] = predictions_df['predicted_value'].values
    
    # 保存预测结果
    predictions_path = os.path.join(outer_cv_dir, f"{base_name}_outer_cv_predictions.csv")
    original_data.to_csv(predictions_path, index=False)
    logging.info(f"Outer CV预测结果已保存: {predictions_path}")
    
    # 保存fold级别的预测结果
    fold_predictions_path = os.path.join(outer_cv_dir, f"{base_name}_outer_cv_fold_predictions.csv")
    predictions_df.to_csv(fold_predictions_path, index=False)
    
    # 为每个fold单独生成散点图
    for fold in range(outer_splits):
        fold_mask = predictions_df['fold'] == fold
        if np.sum(fold_mask) > 0:
            fold_true = predictions_df.loc[fold_mask, 'true_value'].values
            fold_pred = predictions_df.loc[fold_mask, 'predicted_value'].values
            
            # 生成fold级别的散点图
            fold_scatter_dir = os.path.join(outer_cv_dir, "fold_scatter_plots")
            os.makedirs(fold_scatter_dir, exist_ok=True)
            
            plt.figure(figsize=(6, 6))
            ax = plt.gca()
            ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
            
            for spine in ['top', 'right', 'bottom', 'left']:
                ax.spines[spine].set_visible(True)
                ax.spines[spine].set_linewidth(2)
            
            plt.grid(False)
            plt.scatter(fold_true, fold_pred, alpha=0.8, s=70, color=IPHONE_COLORS['scatter'], edgecolors='none')
            
            xymin = float(min(fold_true.min(), fold_pred.min()))
            xymax = float(max(fold_true.max(), fold_pred.max()))
            data_range = xymax - xymin
            pad = data_range * 0.05 if data_range > 0 else 1
            lim_min, lim_max = xymin - pad, xymax + pad
            
            plt.plot([lim_min, lim_max], [lim_min, lim_max], linestyle='--', 
                    color=IPHONE_COLORS['line'], linewidth=3)
            
            r2_fold = r2_score(fold_true, fold_pred)
            mae_fold = mean_absolute_error(fold_true, fold_pred)
            
            plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
            plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')
            
            # 显示fold级别的指标（不显示标准差）
            plt.text(0.05, 0.95, f"R² = {r2_fold:.3f}\nMAE = {mae_fold:.3f}\nFold {fold+1}", 
                    transform=ax.transAxes, verticalalignment='top', fontsize=16, 
                    color=IPHONE_COLORS['text'], bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=5))
            
            plt.xlim([lim_min, lim_max])
            plt.ylim([lim_min, lim_max])
            plt.tight_layout()
            
            fold_scatter_path = os.path.join(fold_scatter_dir, f"{base_name}_fold_{fold+1}_scatter.png")
            plt.savefig(fold_scatter_path, dpi=600, bbox_inches='tight', facecolor='white')
            plt.close()
    
    # 计算Outer CV的平均值和标准差
    metrics_outer = {
        "RMSE": (float(np.mean(rmse_list)), float(np.std(rmse_list))),
        "R2": (float(np.mean(r2_list)), float(np.std(r2_list))),
        "MAE": (float(np.mean(mae_list)), float(np.std(mae_list))),
        # 直接计算总体指标（用于比较）
        "Overall_R2_direct": float(r2_score(np.array(all_true), np.array(all_pred))) if len(all_true) > 0 else None,
        "Overall_MAE_direct": float(mean_absolute_error(np.array(all_true), np.array(all_pred))) if len(all_true) > 0 else None
    }
    
    logging.info(f"Outer CV结果汇总 - RMSE: {metrics_outer['RMSE'][0]:.4f}±{metrics_outer['RMSE'][1]:.4f}")
    logging.info(f"              R²: {metrics_outer['R2'][0]:.4f}±{metrics_outer['R2'][1]:.4f}")
    logging.info(f"              MAE: {metrics_outer['MAE'][0]:.4f}±{metrics_outer['MAE'][1]:.4f}")
    
    # 生成总体Outer CV散点图（高质量格式）- 使用Outer CV平均值±标准差
    logging.info("生成总体Outer CV散点图...")
    all_true_array = np.array(all_true)
    all_pred_array = np.array(all_pred)
    
    # 准备fold指标数据
    fold_metrics_data = {
        "R2": r2_list,
        "MAE": mae_list,
        "RMSE": rmse_list
    }
    
    r2_overall, mae_overall = plot_scatter_and_residuals(
        all_true_array, all_pred_array, 
        outer_cv_dir, base_name, dpi=600,
        outer_cv_metrics=metrics_outer,  # 传入Outer CV指标
        fold_metrics=fold_metrics_data   # 传入fold级别指标
    )
    
    # 更新metrics_outer以包含直接计算的总体指标
    if r2_overall is not None and mae_overall is not None:
        metrics_outer["Overall_R2"] = float(r2_overall)
        metrics_outer["Overall_MAE"] = float(mae_overall)
    
    # Best scheme (在完整数据上重新运行inner CV以获得最佳参数)
    logging.info("在完整数据集上进行最终参数调优...")
    best_global_params = optuna_inner_cv(
        X_all, y_all, base_params, pretrain_model,
        n_trials=n_trials, n_splits=inner_splits, 
        new_trees=new_trees
    )
    final_params = base_params.copy()
    final_params.update(best_global_params)
    
    logging.info(f"最终模型参数: {final_params}")

    # 训练最终模型
    final_model = lgb.LGBMRegressor(
        **final_params,
        n_estimators=pretrain_model.n_estimators + new_trees,
        random_state=SEED,
        force_col_wise=True,
        deterministic=True,
        feature_fraction_seed=SEED,
        bagging_seed=SEED,
        drop_seed=SEED,
        data_random_seed=SEED
    )
    
    logging.info("训练最终模型...")
    final_model.fit(
        X_all, y_all, 
        init_model=pretrain_model,
        callbacks=[log_evaluation(period=0)]  # 不显示日志
    )
    y_pred_all = final_model.predict(X_all)

    # 保存最终模型和文件
    model_path = os.path.join(dataset_dir, f"{base_name}_final_model.joblib")
    joblib.dump(final_model, model_path)
    logging.info(f"最终模型已保存: {model_path}")
    
    # 生成最终模型的散点图（显示直接计算的指标）
    final_scatter_dir = os.path.join(dataset_dir, "final_model_scatter")
    os.makedirs(final_scatter_dir, exist_ok=True)
    
    # 计算最终模型的指标
    r2_final = r2_score(y_all, y_pred_all)
    mae_final = mean_absolute_error(y_all, y_pred_all)
    
    # 为最终模型创建一个简单的指标字典
    final_metrics = {
        "R2": (r2_final, 0),  # 单个模型，标准差为0
        "MAE": (mae_final, 0)
    }
    
    plot_scatter_and_residuals(
        y_all, y_pred_all, 
        final_scatter_dir, f"{base_name}_final", dpi=600,
        outer_cv_metrics=final_metrics  # 传入最终模型指标
    )
    
    # 保存参数文件
    params_path = os.path.join(dataset_dir, f"{base_name}_best_params.json")
    with open(params_path, "w") as f:
        json.dump(final_params, f, indent=4)
    
    logging.info(f"最佳参数已保存: {params_path}")

    # 创建详细报告
    report = {
        "dataset": base_name,
        "data_shape": df.shape,
        "features_used": len(feature_cols),
        "pretrain_trees": pretrain_model.n_estimators,
        "new_trees_added": new_trees,
        "total_trees": pretrain_model.n_estimators + new_trees,
        "outer_cv_metrics": metrics_outer,
        "outer_cv_fold_performance": {
            "RMSE": [float(x) for x in rmse_list],
            "R2": [float(x) for x in r2_list],
            "MAE": [float(x) for x in mae_list]
        },
        "best_params": best_global_params,
        "predictions_saved": predictions_path,
        "model_saved": model_path,
        "overall_samples": len(all_true_array)
    }
    
    # 保存指标报告
    metrics_path = os.path.join(dataset_dir, f"{base_name}_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(report, f, indent=4)
    
    logging.info(f"指标报告已保存: {metrics_path}")
    logging.info(f"=== 数据集 {base_name} 的嵌套CV完成 ===")
    
    return report


# -------------------- Main Loop --------------------
if __name__ == "__main__":
    # 设置全局随机种子
    logging.info(f"=== 开始迁移学习嵌套CV过程 ===")
    logging.info(f"随机种子: {SEED}")
    logging.info(f"数据目录: {DATA_DIR}")
    logging.info(f"预训练目录: {PRETRAIN_DIR}")
    logging.info(f"输出目录: {OUTPUT_DIR}")
    logging.info(f"目标数据集: {TARGET_DATASETS}")
    
    # 加载预训练资产
    pretrain_model, feature_cols, imputer, pretrain_params = load_pretrained_assets()
    
    results_all = {}
    summary_rows = []

    for file_name in TARGET_DATASETS:
        try:
            base_name = file_name.replace('.csv', '')
            logging.info(f"\n{'='*60}")
            logging.info(f"开始处理数据集: {base_name}")
            logging.info(f"{'='*60}")
            
            # 重置随机种子以确保一致性
            random.seed(SEED)
            np.random.seed(SEED)
            
            # 记录数据集基本信息
            data_path = os.path.join(DATA_DIR, file_name)
            if not os.path.exists(data_path):
                logging.error(f"数据文件不存在: {data_path}")
                continue
                
            df = pd.read_csv(data_path)
            logging.info(f"数据集基本信息:")
            logging.info(f"  - 数据路径: {data_path}")
            logging.info(f"  - 数据形状: {df.shape}")
            logging.info(f"  - 特征数量: {len(feature_cols)}")
            logging.info(f"  - 预训练模型树数量: {pretrain_model.n_estimators}")
            
            # 执行嵌套CV
            metrics = nested_cv_finetune(
                file_name, 
                pretrain_model, 
                feature_cols, 
                imputer, 
                pretrain_params
            )
            results_all[base_name] = metrics
            
            # 提取性能指标
            rmse_mean, rmse_std = metrics["outer_cv_metrics"]["RMSE"]
            r2_mean, r2_std = metrics["outer_cv_metrics"]["R2"]
            mae_mean, mae_std = metrics["outer_cv_metrics"]["MAE"]
            overall_r2 = metrics["outer_cv_metrics"].get("Overall_R2", r2_mean)
            overall_mae = metrics["outer_cv_metrics"].get("Overall_MAE", mae_mean)
            
            summary_rows.append({
                "Dataset": base_name,
                "RMSE_mean": rmse_mean,
                "RMSE_std": rmse_std,
                "R2_mean": r2_mean,
                "R2_std": r2_std,
                "MAE_mean": mae_mean,
                "MAE_std": mae_std,
                "Overall_R2": overall_r2,
                "Overall_MAE": overall_mae,
                "N_samples": metrics.get("overall_samples", "N/A")
            })
            
            logging.info(f"数据集 {base_name} 处理完成!")
            logging.info(f"整体性能 - R²: {overall_r2:.4f}, MAE: {overall_mae:.4f}")
            
        except Exception as e:
            base_name = file_name.replace('.csv', '')
            logging.error(f"[{base_name}] 处理失败: {str(e)}", exc_info=True)
            results_all[base_name] = {"status": "fail", "message": str(e)}

    # Save summary files
    logging.info("\n=== 生成总结报告 ===")
    
    # 保存所有指标的JSON总结
    summary_json_path = os.path.join(OUTPUT_DIR, "all_metrics_summary.json")
    with open(summary_json_path, "w") as f:
        json.dump(results_all, f, indent=4)
    logging.info(f"所有指标JSON总结已保存: {summary_json_path}")

    # 生成CSV总结
    if summary_rows:
        df_summary = pd.DataFrame(summary_rows)
        summary_csv_path = os.path.join(OUTPUT_DIR, "all_metrics_summary.csv")
        df_summary.to_csv(summary_csv_path, index=False)
        logging.info(f"CSV总结已保存: {summary_csv_path}")
        
        # 生成性能对比图
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # R²对比
        datasets = [row["Dataset"] for row in summary_rows]
        r2_values = [row["Overall_R2"] for row in summary_rows]
        
        axes[0].bar(datasets, r2_values, color=IPHONE_COLORS['scatter'])
        axes[0].set_xlabel('Dataset', fontweight='bold')
        axes[0].set_ylabel('R² Score', fontweight='bold')
        axes[0].set_title('Outer CV R² Score Comparison', fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(True, alpha=0.3)
        
        # MAE对比
        mae_values = [row["Overall_MAE"] for row in summary_rows]
        axes[1].bar(datasets, mae_values, color=IPHONE_COLORS['line'])
        axes[1].set_xlabel('Dataset', fontweight='bold')
        axes[1].set_ylabel('MAE (s)', fontweight='bold')
        axes[1].set_title('Outer CV MAE Comparison', fontweight='bold')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        comparison_path = os.path.join(OUTPUT_DIR, "outer_cv_performance_comparison.png")
        plt.savefig(comparison_path, dpi=600, bbox_inches='tight')
        plt.close()
        
        logging.info(f"性能对比图已保存: {comparison_path}")

    logging.info("=== 迁移学习嵌套CV过程全部完成 ===")
    print(f"\n✅ 所有处理完成！")
    print(f"📊 结果目录: {OUTPUT_DIR}")
    print(f"📈 每个数据集的详细结果保存在各自的子目录中")
    print(f"📋 总结文件: {os.path.join(OUTPUT_DIR, 'all_metrics_summary.csv')}")
    print(f"🎯 Outer CV预测结果保存在每个数据集的 'outer_cv_predictions' 子目录中")

/home/xuxianyan/anaconda3-1/envs/fusion_env1/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/home/xuxianyan/anaconda3-1/envs/fusion_env1/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-02-11 20:21:32,861] A new study created in memory with name: no-name-c7df3227-a84a-4597-9e70-64e1e035c013
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/


✅ 所有处理完成！
📊 结果目录: ./2-lgb-TL-models-other4
📈 每个数据集的详细结果保存在各自的子目录中
📋 总结文件: ./2-lgb-TL-models-other4/all_metrics_summary.csv
🎯 Outer CV预测结果保存在每个数据集的 'outer_cv_predictions' 子目录中
